In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "openai-community/gpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

model.eval()

print("模型加载成功！")

/home/navy/projects/repos/github/navys-courses/llm/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 148/148 [00:00<00:00, 1704.02it/s]


模型加载成功！


In [4]:
print(model)

# Total parameters
total_params = sum(p.numel() for p in model.parameters())

# Trainable parameters
trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Total parameters (Millions): {total_params / 1e6:.2f} M")

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)
Total parameters: 12

In [5]:
config = model.config

print("Vocabulary size:", config.vocab_size)
print("Maximum sequence length:", config.n_positions)
print("Hidden dimension:", config.n_embd)
print("Number of Transformer layers:", config.n_layer)
print("Number of Attention heads:", config.n_head)
print("Feed-forward dimension:", config.n_inner or config.n_embd * 4)

Vocabulary size: 50257
Maximum sequence length: 1024
Hidden dimension: 768
Number of Transformer layers: 12
Number of Attention heads: 12
Feed-forward dimension: 3072


In [3]:
# 输入一句话
prompt = "The capital of France is"

# 将文本转换为 Token
inputs = tokenizer(prompt, return_tensors="pt")

# 模型前向计算
with torch.no_grad():
    outputs = model(**inputs)

# 取最后一个位置的 Logits
logits = outputs.logits[0, -1, :]

# Softmax：将 Logits 转换为概率
probabilities = torch.softmax(logits, dim=-1)

# 找出概率最高的 10 个 Token
top_probs, top_indices = torch.topk(probabilities, 10)

print("输入文本：", prompt)
print("\n预测概率最高的 10 个 Token：\n")

for i in range(10):
    token_id = top_indices[i].item()

    token = tokenizer.decode([token_id])
    logit = logits[token_id].item()
    probability = top_probs[i].item()

    print(
        f"Token: {repr(token):15s} "
        f"Logit: {logit:10.4f} "
        f"Probability: {probability:.6%}"
    )

输入文本： The capital of France is

预测概率最高的 10 个 Token：

Token: ' the'          Logit:  -100.2498 Probability: 8.459118%
Token: ' now'          Logit:  -100.8175 Probability: 4.794700%
Token: ' a'            Logit:  -100.8555 Probability: 4.616052%
Token: ' France'       Logit:  -101.2101 Probability: 3.237803%
Token: ' Paris'        Logit:  -101.2143 Probability: 3.224466%
Token: ' in'           Logit:  -101.4052 Probability: 2.664055%
Token: ' also'         Logit:  -101.4141 Probability: 2.640441%
Token: ' not'          Logit:  -101.5168 Probability: 2.382802%
Token: ' home'         Logit:  -101.5371 Probability: 2.334808%
Token: ' still'        Logit:  -101.9462 Probability: 1.550850%


In [6]:
print("Input IDs shape:", inputs["input_ids"].shape)
print("Logits shape:", outputs.logits.shape)

Input IDs shape: torch.Size([1, 5])
Logits shape: torch.Size([1, 5, 50257])


In [7]:
from transformers import pipeline, set_seed
generator = pipeline('text-generation', model='gpt2')
set_seed(42)
generator(prompt, max_length=30, num_return_sequences=5)

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 1391.29it/s]


[{'generated_text': "The capital of France is home to France's oldest public libraries, the Royal Library of Canada, the Royal National Library of Canada and the French National Library"},
 {'generated_text': 'The capital of France is France, which is not a country of the capital, but a city of the capital. The capital must be a city of'},
 {'generated_text': 'The capital of France is Paris, and the capital of the French people is Marseille. I am a realist, for my country is a very'},
 {'generated_text': 'The capital of France is the former French colony of Mauritius and Morocco.\n\n(AFP Photo/Riccardo Bensch)\n'},
 {'generated_text': "The capital of France is still in mourning. When he died in September, the French government announced a new curfew for all citizens. The city's mayor"}]

In [8]:
# Calculate model parameter memory
model_size_bytes = sum(
    p.numel() * p.element_size()
    for p in model.parameters()
)

print(f"Model size: {model_size_bytes:,} bytes")
print(f"Model size: {model_size_bytes / 1e6:.2f} MB")
print(f"Model size: {model_size_bytes / (1024**2):.2f} MiB")

Model size: 497,759,232 bytes
Model size: 497.76 MB
Model size: 474.70 MiB


In [9]:
for name, param in model.named_parameters():
    print(name, param.dtype, param.element_size())
    break

transformer.wte.weight torch.float32 4


In [ ]:
import os
from huggingface_hub import snapshot_download

model_path = snapshot_download(model_name)

for filename in os.listdir(model_path):
    filepath = os.path.join(model_path, filename)

    if os.path.isfile(filepath):
        size_mb = os.path.getsize(filepath) / 1e6
        print(f"{filename:40s} {size_mb:10.2f} MB")

Fetching 26 files:  81%|████████  | 21/26 [06:37<01:51, 22.33s/it]